# Moindres carrés

Dans ce notebook, je compare les différentes façons de résoudre un problème de moindres carrés linéaire, et je regarde pourquoi elles ne se valent pas numériquement. D'abord un rappel de cours, puis cinq exercices corrigés.

---

## Rappel de cours

### Le problème des moindres carrés

Soit $A \in \mathcal{M}_{n,p}(\mathbb{R})$ avec $n \geq p$ (système surdéterminé) et $b \in \mathbb{R}^n$.

On cherche $\hat{x} \in \mathbb{R}^p$ qui minimise :
$$\hat{x} = \arg\min_{x \in \mathbb{R}^p} \|Ax - b\|_2^2$$

### Interprétation géométrique

$A\hat{x}$ est la projection orthogonale de $b$ sur $\text{Im}(A)$ : c'est le point de l'image de $A$ le plus proche de $b$. Le résidu $r = b - A\hat{x}$ est donc orthogonal à $\text{Im}(A)$, autrement dit $r \in \text{Ker}(A^\top)$.

### Équations normales

En écrivant cette orthogonalité, on obtient les équations normales :
$$\boxed{A^\top A \hat{x} = A^\top b}$$

Si $A$ est de rang plein colonnes ($\text{rg}(A) = p$), alors $A^\top A$ est inversible et :
$$\hat{x} = (A^\top A)^{-1} A^\top b$$

### Pseudo-inverse de Moore-Penrose

Dans le cas de rang plein, $A^+ = (A^\top A)^{-1} A^\top$ et $\hat{x} = A^+ b$. Dans le cas général, on la définit via la SVD, et elle donne la solution de norme minimale.

### Méthodes de résolution

Les trois donnent le même $\hat{x}$ en arithmétique exacte. La différence est numérique : former $A^\top A$ élève le conditionnement au carré.

| Méthode | Formule | Complexité | Stabilité |
|---------|---------|------------|----------|
| Équations normales | $(A^\top A)^{-1} A^\top b$ | $O(np^2 + p^3)$ | Faible (cond²) |
| QR | $R^{-1} Q^\top b$ | $O(np^2)$ | Bonne |
| SVD | $V \Sigma^+ U^\top b$ | $O(np^2)$ | Excellente |

In [1]:
# Imports
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.linalg import solve_triangular, qr, svd

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)

---

## Méthodes implémentées

In [2]:
def least_squares_normal(A, b):
    """Moindres carrés via équations normales."""
    AtA = A.T @ A
    Atb = A.T @ b
    return np.linalg.solve(AtA, Atb)

def least_squares_qr(A, b):
    """Moindres carrés via factorisation QR."""
    Q, R = np.linalg.qr(A)
    return solve_triangular(R, Q.T @ b)

def least_squares_svd(A, b):
    """Moindres carrés via SVD (pseudo-inverse)."""
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    # x = V @ Σ⁺ @ U.T @ b
    return Vt.T @ (np.diag(1/S) @ (U.T @ b))

---

## Exercices

### Exercice 1 (facile) : Régression linéaire simple

On veut ajuster une droite $y = ax + b$ aux données :

| $x$ | 0 | 1 | 2 | 3 |
|-----|---|---|---|---|
| $y$ | 1 | 2 | 2 | 4 |

À faire :
1. Écrire le problème sous forme matricielle $A\beta \approx y$ avec $\beta = (b, a)^\top$
2. Résoudre par les équations normales
3. Tracer la droite de régression

In [3]:
# À FAIRE : Exercice 1

x = np.array([0, 1, 2, 3])
y = np.array([1, 2, 2, 4])

# 1. Construction de A
# y = b + a*x => [1, x] @ [b, a]^T = y
A = None  # À compléter : np.column_stack([np.ones_like(x), x])

# 2. Équations normales
beta = None  # À compléter

# 3. Visualisation

<details>
<summary><b>Correction Exercice 1</b></summary>
</details>

In [4]:
# CORRECTION Exercice 1

x = np.array([0, 1, 2, 3], dtype=float)
y = np.array([1, 2, 2, 4], dtype=float)

print("Données :")
print(f"x = {x}")
print(f"y = {y}")

# 1. Matrice A
A = np.column_stack([np.ones_like(x), x])
print("\n1. Matrice A (colonne de 1, colonne de x) :")
print(A)

# 2. Équations normales
print("\n2. Équations normales : A^T A β = A^T y")
AtA = A.T @ A
Aty = A.T @ y
print(f"   A^T A =\n{AtA}")
print(f"\n   A^T y = {Aty}")

beta = np.linalg.solve(AtA, Aty)
b_intercept, a_slope = beta
print(f"\n   Solution : β = {beta}")
print(f"   Droite : y = {b_intercept:.4f} + {a_slope:.4f}x")

# 3. Visualisation
x_line = np.linspace(-0.5, 3.5, 100)
y_line = b_intercept + a_slope * x_line

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Données', marker=dict(size=12)))
fig.add_trace(go.Scatter(x=x_line, y=y_line, mode='lines', name=f'y = {b_intercept:.2f} + {a_slope:.2f}x'))

# Résidus
y_pred = b_intercept + a_slope * x
for xi, yi, ypi in zip(x, y, y_pred):
    fig.add_trace(go.Scatter(x=[xi, xi], y=[yi, ypi], mode='lines',
                             line=dict(color='red', dash='dash'), showlegend=False))

fig.update_layout(title='Régression linéaire par moindres carrés',
                  xaxis_title='x', yaxis_title='y', width=600, height=400)
fig.show()

# Erreur
residual = np.linalg.norm(y - y_pred)
print(f"\n   Résidu ||y - Aβ|| = {residual:.4f}")

Données :
x = [0. 1. 2. 3.]
y = [1. 2. 2. 4.]

1. Matrice A (colonne de 1, colonne de x) :
[[1. 0.]
 [1. 1.]
 [1. 2.]
 [1. 3.]]

2. Équations normales : A^T A β = A^T y
   A^T A =
[[ 4.  6.]
 [ 6. 14.]]

   A^T y = [ 9. 18.]

   Solution : β = [0.9 0.9]
   Droite : y = 0.9000 + 0.9000x



   Résidu ||y - Aβ|| = 0.8367


### Exercice 2 (intermédiaire) : Comparaison des méthodes

Générer un problème de moindres carrés et comparer les 3 méthodes.

À faire :
1. Générer $A$ de taille $100 \times 10$ et $b = Ax_{true} + \epsilon$ avec bruit
2. Résoudre avec les 3 méthodes
3. Comparer les erreurs $\|\hat{x} - x_{true}\|$

In [5]:
# À FAIRE : Exercice 2

n, p = 100, 10
np.random.seed(42)

# Données
A = np.random.randn(n, p)
x_true = np.random.randn(p)
noise = 0.1 * np.random.randn(n)
b = A @ x_true + noise

# Résolution par les 3 méthodes
x_normal = None  # À compléter
x_qr = None  # À compléter
x_svd = None  # À compléter

# Erreurs

<details>
<summary><b>Correction Exercice 2</b></summary>
</details>

In [6]:
# CORRECTION Exercice 2

n, p = 100, 10
np.random.seed(42)

# Génération des données
A = np.random.randn(n, p)
x_true = np.random.randn(p)
noise = 0.1 * np.random.randn(n)
b = A @ x_true + noise

print(f"Problème : A de taille {n}×{p}")
print(f"Niveau de bruit : σ = 0.1")

# Résolution
x_normal = least_squares_normal(A, b)
x_qr = least_squares_qr(A, b)
x_svd = least_squares_svd(A, b)
x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]  # référence numpy

# Erreurs
errors = {
    'Équations normales': np.linalg.norm(x_normal - x_true),
    'QR': np.linalg.norm(x_qr - x_true),
    'SVD': np.linalg.norm(x_svd - x_true),
    'np.linalg.lstsq': np.linalg.norm(x_lstsq - x_true)
}

print("\nErreur ||x̂ - x_true|| pour chaque méthode :")
for method, err in errors.items():
    print(f"  {method:25s} : {err:.6f}")

# Résidus
residuals = {
    'Équations normales': np.linalg.norm(A @ x_normal - b),
    'QR': np.linalg.norm(A @ x_qr - b),
    'SVD': np.linalg.norm(A @ x_svd - b),
}

print("\nRésidu ||Ax̂ - b|| :")
for method, res in residuals.items():
    print(f"  {method:25s} : {res:.6f}")

print("\n=> Les 3 méthodes donnent le même résultat (aux erreurs numériques près)")

Problème : A de taille 100×10
Niveau de bruit : σ = 0.1

Erreur ||x̂ - x_true|| pour chaque méthode :
  Équations normales        : 0.034759
  QR                        : 0.034759
  SVD                       : 0.034759
  np.linalg.lstsq           : 0.034759

Résidu ||Ax̂ - b|| :
  Équations normales        : 0.980280
  QR                        : 0.980280
  SVD                       : 0.980280

=> Les 3 méthodes donnent le même résultat (aux erreurs numériques près)


### Exercice 3 (intermédiaire) : Interprétation géométrique

Visualiser la projection orthogonale en 2D.

Soit $A = \begin{pmatrix} 1 \\ 2 \end{pmatrix}$ (une seule colonne) et $b = \begin{pmatrix} 3 \\ 1 \end{pmatrix}$.

À faire :
1. Calculer $\hat{x}$ et $A\hat{x}$ (projection de $b$ sur $\text{Im}(A) = \text{Vect}(a)$)
2. Vérifier que $r = b - A\hat{x} \perp a$
3. Visualiser

In [7]:
# À FAIRE : Exercice 3

A = np.array([[1], [2]], dtype=float)
b = np.array([3, 1], dtype=float)

# 1. Solution
x_hat = None  # À compléter : (A.T @ A)^{-1} A.T @ b
projection = None  # A @ x_hat

# 2. Résidu
r = None  # b - projection
# Vérifier <r, a> = 0

<details>
<summary><b>Correction Exercice 3</b></summary>
</details>

In [8]:
# CORRECTION Exercice 3

A = np.array([[1], [2]], dtype=float)
b = np.array([3, 1], dtype=float)
a = A.flatten()  # vecteur colonne de A

print(f"a = {a}")
print(f"b = {b}")

# 1. Solution
AtA = A.T @ A  # = ||a||² = 5
Atb = A.T @ b  # = <a, b> = 5
x_hat = float(Atb / AtA)  # = 1

projection = x_hat * a  # = (1, 2)

print(f"\n1. Solution :")
print(f"   A^T A = {AtA[0,0]} = ||a||²")
print(f"   A^T b = {Atb[0]} = <a, b>")
print(f"   x̂ = <a,b>/||a||² = {x_hat}")
print(f"   Projection Ax̂ = {projection}")

# 2. Résidu
r = b - projection
print(f"\n2. Résidu :")
print(f"   r = b - Ax̂ = {r}")
print(f"   <r, a> = {np.dot(r, a):.2e}")
print(f"   r ⊥ a ? {np.isclose(np.dot(r, a), 0)} ✓")

# 3. Visualisation
fig = go.Figure()

# Vecteur a (direction de Im(A))
fig.add_trace(go.Scatter(x=[0, a[0]], y=[0, a[1]], mode='lines+markers',
                         name='a (Im(A))', line=dict(width=3, color='blue')))

# Vecteur b
fig.add_trace(go.Scatter(x=[0, b[0]], y=[0, b[1]], mode='lines+markers',
                         name='b', line=dict(width=3, color='red')))

# Projection
fig.add_trace(go.Scatter(x=[0, projection[0]], y=[0, projection[1]], mode='lines+markers',
                         name='Ax̂ (projection)', line=dict(width=3, color='green')))

# Résidu (perpendiculaire)
fig.add_trace(go.Scatter(x=[projection[0], b[0]], y=[projection[1], b[1]], mode='lines',
                         name='r = b - Ax̂ (résidu)', line=dict(width=2, color='orange', dash='dash')))

fig.update_layout(title='Interprétation géométrique des moindres carrés',
                  xaxis_title='x₁', yaxis_title='x₂',
                  width=600, height=500)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.show()

print("\nInterprétation : Ax̂ est la projection orthogonale de b sur Im(A)")

a = [1. 2.]
b = [3. 1.]

1. Solution :
   A^T A = 5.0 = ||a||²
   A^T b = 5.0 = <a, b>
   x̂ = <a,b>/||a||² = 1.0
   Projection Ax̂ = [1. 2.]

2. Résidu :
   r = b - Ax̂ = [ 2. -1.]
   <r, a> = 0.00e+00
   r ⊥ a ? True ✓


/var/folders/7q/8z0gkbws6cj41q4k8kf9x2kc0000gn/T/ipykernel_6413/3273414061.py:13: DeprecationWarning:

Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)




Interprétation : Ax̂ est la projection orthogonale de b sur Im(A)


### Exercice 4 (avancé) : Conditionnement

Le **conditionnement** $\kappa(A) = \|A\| \cdot \|A^+\| = \sigma_1 / \sigma_r$ mesure la sensibilité aux perturbations.

À faire :
1. Créer une matrice bien conditionnée ($\kappa \approx 1$) et une mal conditionnée ($\kappa \gg 1$)
2. Ajouter une petite perturbation au second membre $b$
3. Observer l'amplification de l'erreur

In [9]:
# À FAIRE : Exercice 4

n, p = 50, 5
np.random.seed(0)

# Matrice bien conditionnée
A_good = np.random.randn(n, p)

# Matrice mal conditionnée (colonnes presque colinéaires)
A_bad = np.random.randn(n, p)
A_bad[:, 1] = A_bad[:, 0] + 1e-6 * np.random.randn(n)  # col 1 ≈ col 0

# Calculer les conditionnements
cond_good = None  # np.linalg.cond(A_good)
cond_bad = None  # np.linalg.cond(A_bad)

<details>
<summary><b>Correction Exercice 4</b></summary>
</details>

In [10]:
# CORRECTION Exercice 4

n, p = 50, 5
np.random.seed(0)

# Matrice bien conditionnée
A_good = np.random.randn(n, p)

# Matrice mal conditionnée
A_bad = np.random.randn(n, p)
A_bad[:, 1] = A_bad[:, 0] + 1e-8 * np.random.randn(n)

# Conditionnements
cond_good = np.linalg.cond(A_good)
cond_bad = np.linalg.cond(A_bad)

print(f"Conditionnement :")
print(f"  A_good : κ = {cond_good:.2f}")
print(f"  A_bad  : κ = {cond_bad:.2e}")

# Test de sensibilité
print("\nTest de sensibilité aux perturbations :")

for name, A in [('A_good', A_good), ('A_bad', A_bad)]:
    print(f"\n{name} :")
    
    # Solution exacte
    x_true = np.random.randn(p)
    b = A @ x_true
    
    # Perturbation
    delta_b = 1e-10 * np.random.randn(n)
    b_pert = b + delta_b
    
    # Solutions
    x_clean = np.linalg.lstsq(A, b, rcond=None)[0]
    x_pert = np.linalg.lstsq(A, b_pert, rcond=None)[0]
    
    rel_pert_b = np.linalg.norm(delta_b) / np.linalg.norm(b)
    rel_error_x = np.linalg.norm(x_pert - x_clean) / np.linalg.norm(x_clean)
    amplification = rel_error_x / rel_pert_b if rel_pert_b > 0 else 0
    
    print(f"  Perturbation relative de b : {rel_pert_b:.2e}")
    print(f"  Erreur relative sur x     : {rel_error_x:.2e}")
    print(f"  Amplification             : {amplification:.2f}×")
    print(f"  (κ = {np.linalg.cond(A):.2e})")

Conditionnement :
  A_good : κ = 1.53
  A_bad  : κ = 1.85e+08

Test de sensibilité aux perturbations :

A_good :
  Perturbation relative de b : 9.25e-11
  Erreur relative sur x     : 2.51e-11
  Amplification             : 0.27×
  (κ = 1.53e+00)

A_bad :
  Perturbation relative de b : 4.62e-11
  Erreur relative sur x     : 9.00e-04
  Amplification             : 19486706.92×
  (κ = 1.85e+08)


### Exercice 5 (avancé) : Régression polynomiale

Ajuster un polynôme de degré $d$ aux données.

À faire :
1. Générer des données bruitées à partir de $y = \sin(x)$
2. Ajuster des polynômes de degrés 1, 3, 5, 10, 15
3. Observer le **sur-apprentissage** (overfitting) pour les hauts degrés

In [11]:
# À FAIRE : Exercice 5

np.random.seed(42)
n_points = 20

x_data = np.linspace(0, 2*np.pi, n_points)
y_true = np.sin(x_data)
y_data = y_true + 0.2 * np.random.randn(n_points)

# Pour un polynôme de degré d, la matrice de Vandermonde est :
# A = [1, x, x², ..., x^d]
def vandermonde(x, degree):
    return np.column_stack([x**k for k in range(degree + 1)])

<details>
<summary><b>Correction Exercice 5</b></summary>
</details>

In [16]:
# CORRECTION Exercice 5

np.random.seed(42)
n_points = 20

x_data = np.linspace(0, 2*np.pi, n_points)
y_true = np.sin(x_data)
y_data = y_true + 0.2 * np.random.randn(n_points)

def vandermonde(x, degree):
    return np.column_stack([x**k for k in range(degree + 1)])

# Grille fine pour le tracé
x_fine = np.linspace(0, 2*np.pi, 200)

fig = go.Figure()

# Données
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers',
                         name='Données bruitées', marker=dict(size=8, color='black')))

# Vraie fonction
fig.add_trace(go.Scatter(x=x_fine, y=np.sin(x_fine), mode='lines',
                         name='sin(x) (vrai)', line=dict(dash='dash', color='gray')))

# Polynômes de différents degrés
degrees = [1, 3, 5, 10, 15,16]
colors = px.colors.qualitative.Set1

print("Régression polynomiale :")
print(f"{'Degré':>6} | {'Erreur train':>12} | {'Conditionnement':>15}")
print("-" * 40)

for i, d in enumerate(degrees):
    A = vandermonde(x_data, d)
    coeffs = np.linalg.lstsq(A, y_data, rcond=None)[0]
    
    # Prédiction
    y_pred_train = vandermonde(x_data, d) @ coeffs
    y_pred_fine = vandermonde(x_fine, d) @ coeffs
    
    # Erreur d'entraînement
    train_error = np.linalg.norm(y_data - y_pred_train) / np.sqrt(n_points)
    cond = np.linalg.cond(A)
    
    print(f"{d:>6} | {train_error:>12.4f} | {cond:>15.2e}")
    
    fig.add_trace(go.Scatter(x=x_fine, y=y_pred_fine, mode='lines',
                             name=f'Degré {d}', line=dict(color=colors[i % len(colors)])))

fig.update_layout(title='Régression polynomiale et sur-apprentissage',
                  xaxis_title='x', yaxis_title='y',
                  width=800, height=500, yaxis_range=[-2, 2])
fig.show()

print("\nObservation : degré élevé => sur-apprentissage + mauvais conditionnement")

Régression polynomiale :
 Degré | Erreur train | Conditionnement
----------------------------------------
     1 |       0.5499 |        7.47e+00
     3 |       0.1823 |        5.90e+02
     5 |       0.1308 |        7.30e+04
    10 |       0.1078 |        2.85e+10
    15 |       0.0888 |        4.76e+16
    16 |       0.0936 |        1.10e+18



Observation : degré élevé => sur-apprentissage + mauvais conditionnement


---

## Ce que je retiens

- Le problème : $\min_x \|Ax - b\|^2$ avec $A \in \mathbb{R}^{n \times p}$ et $n > p$. Géométriquement, $A\hat{x}$ est la projection de $b$ sur $\text{Im}(A)$.
- Les équations normales $A^\top A \hat{x} = A^\top b$ sont les plus simples à écrire, mais les moins stables.
- QR ($\hat{x} = R^{-1} Q^\top b$) est le bon choix par défaut.
- La SVD ($\hat{x} = V \Sigma^+ U^\top b$) est la plus robuste, et c'est la seule qui marche quand $A$ n'est pas de rang plein.
- Le conditionnement $\kappa(A) = \sigma_1/\sigma_r$ dit à quel point une petite erreur sur $b$ peut être amplifiée dans la solution.